In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import TypedDict, Annotated, Literal
from operator import add
from langchain_groq import ChatGroq
from dotenv import load_dotenv,find_dotenv
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage
from langgraph.types import interrupt
from pydantic import BaseModel
import os

In [2]:
load_dotenv(find_dotenv(),override=True)

RAPID_API_KEY = os.getenv('RAPID_API_KEY')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

print(f"Rapid Key Found: {RAPID_API_KEY is not None}")
print(f"Gemini Key Found: {GOOGLE_API_KEY is not None}")
print(f"GROQ Key Found: {GROQ_API_KEY is not None}")

Rapid Key Found: True
Gemini Key Found: True
GROQ Key Found: True


In [3]:


# Set GROQ_API_KEY in environment or pass directly
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
)


# practice 1

In [20]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [41]:
def llm_node(state:State):
    response=llm.invoke(state['messages'])
    return {'messages':response}

def decide_continue(state:State):

    last_msg=state['messages'][-1]
    
    text=last_msg.content.lower()
    if isinstance(text, AIMessage):         # ya msg.__class__.__name__ == "AIMessage"
        print(text)
    if 'no' in text or 'end' in text or 'stop' in text:
        return "__end__"
    
    return 'continue'

In [42]:
workflow = StateGraph(State)
workflow.add_node('llm',llm_node)
workflow.add_conditional_edges(
    'llm',
    decide_continue,
    {
        'continue':'llm',
        '__end__':END
    }
)

workflow.add_edge(START,'llm')
checkpointer=InMemorySaver()
app=workflow.compile(checkpointer=checkpointer)


In [ ]:
thread = {"configurable": {"thread_id": "user_123"}}
while True:
    user_input=input('You: ')
    result=app.invoke({'messages':user_input},config=thread)
    print(user_input)
    if user_input.lower() =='no':
        break



Hey


In [ ]:
# for spcific check[point_id
# get the latest state snapshot
config={'configurable':{'thread_id':'1','checkpoint_id':'1f117cde-57eb-605d-8002-79c328c2b128'}}
workflow.get_state(config)

StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f117cde-57eb-605d-8002-79c328c2b128'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-04T13:27:31.121570+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f117cde-57e8-6a38-8001-87596073eb3d'}}, tasks=(), interrupts=())

In [16]:
config={'configurable':{'thread_id':'1'}}
list(graph.get_state_history(config))

[StateSnapshot(values={'foo': 'b', 'bar': ['a', 'b']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f117cde-57eb-605d-8002-79c328c2b128'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-04T13:27:31.121570+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f117cde-57e8-6a38-8001-87596073eb3d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'foo': 'a', 'bar': ['a']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f117cde-57e8-6a38-8001-87596073eb3d'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-04T13:27:31.120594+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f117cde-57e3-6c30-8000-60cb493f3e07'}}, tasks=(PregelTask(id='ab8defdb-872a-37a5-d2b6-b0e229cbb2d2', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts

In [17]:
config = {"configurable": {"thread_id": "1", "checkpoint_id": "1f117cde-57e8-6a38-8001-87596073eb3d"}}
graph.invoke(None, config=config)

{'foo': 'b', 'bar': ['a', 'b']}

In [28]:

class MyState(BaseModel):
    topic: str
    joke: str = ""

def refine_topic(state:MyState):

    prompt=f"generate a joke on:- {state.topic}"
    respone=llm.invoke(prompt)
    return {'joke':respone.content}


In [29]:
graph = (
  StateGraph(MyState)
  .add_node(refine_topic)
  .add_edge(START, "refine_topic")
  .compile()
)

In [31]:
for chunk,metadata in graph.stream(
    {'topic':'ice cream'},
    stream_mode='messages'#updates # values
):
    if chunk.content:
        print(chunk.content)

Why
 did
 the
 ice
 cream
 go
 to
 therapy
?


Because
 it
 had
 a
 lot
 of
 "
mel
td
owns
"
 and
 was
 struggling
 to
 "
cope
"
 with
 the
 pressure
!
